# Zone analysis — Teselado

Elbow k-selection and K-Means vs Fuzzy C-Means comparison at the same k.
Both methods use identical haversine simulation parameters.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from teselado.config import Settings
from teselado.ingest.loaders import load_orders, load_orders_df, load_restaurants_df
from teselado.clustering.kmeans import KMeans
from teselado.clustering.selector import select_k
from teselado.simulation.compare import compare_clustering_methods
from teselado.simulation.engine import SimulationParams

DATA_DIR = Path('../data/sample')
points = load_orders(DATA_DIR)
orders_df = load_orders_df(DATA_DIR)
restaurants_df = load_restaurants_df(DATA_DIR)
points.shape

In [ ]:
k_min, k_max = 3, 8
k_values = list(range(k_min, k_max))
inertias = []
for k in k_values:
    model = KMeans(k=k).fit(points)
    labels = model.predict(points)
    inertia = sum(
        ((points[labels == label] - model.centroids_[label]) ** 2).sum()
        for label in range(k)
        if (labels == label).any()
    )
    inertias.append(float(inertia))

selected_k = select_k(points, k_min=k_min, k_max=k_max)
plt.plot(k_values, inertias, marker='o')
plt.axvline(selected_k, color='orange', linestyle='--', label=f'elbow k={selected_k}')
plt.title('Elbow plot (WCSS)')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.legend()
plt.show()
selected_k

In [ ]:
comparisons = compare_clustering_methods(
    orders_df,
    restaurants_df,
    k=selected_k,
    methods=['kmeans', 'fuzzy'],
    params=SimulationParams(num_couriers=Settings().num_couriers),
)

rows = []
for item in comparisons:
    m = item.metrics
    rows.append(
        {
            'method': item.method,
            'k': item.k,
            'avg_delivery_min': m['avg_delivery_time_min'],
            'sla_hit_rate': m['sla_hit_rate'],
            'orders_per_hour': m['orders_per_hour'],
            'courier_utilisation': m['courier_utilisation'],
            'boundary_ambiguity': m.get('boundary_ambiguity', {}).get('boundary_point_ratio'),
        }
    )

pd.DataFrame(rows)